<a href="https://colab.research.google.com/github/ToshiroHJJZ/Detection-and-Classification-of-Linguistic-Features-for-Effective-Spam-Identification/blob/main/Adversial_Attack_White_Box.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("/content/drive/MyDrive/HThesis/spam.csv", encoding='latin-1')

In [3]:
#remove the 3 and 4rth column
df = df.drop(df.columns[[2,3,4]], axis=1)
# select only ham part
df=df[df['v1']=='ham']

In [4]:
data= df.rename(columns={'v2': 'text'})

In [5]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification, AdamW

# Load pre-trained BERT model and tokenizer
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    embeddings = model.get_input_embeddings()(inputs["input_ids"]).detach()
    embeddings.requires_grad = True
    return inputs, embeddings

def calculate_gradients(embeddings, inputs, target_label):
    outputs = model(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"])
    loss = F.cross_entropy(outputs.logits, torch.tensor([target_label]))
    loss.backward()
    return embeddings.grad.data

def gradient_based_attack(text, label, epsilon=0.1):
    """Perturb the input text using gradient-based attack."""
    inputs, embeddings = get_bert_embeddings(text)
    gradients = calculate_gradients(embeddings, inputs, label)

    input_ids = inputs["input_ids"]  # Get input_ids from inputs
    embeddings = model.get_input_embeddings()(input_ids).detach()
    embeddings.requires_grad = True
    outputs = model(inputs_embeds=embeddings, attention_mask=inputs["attention_mask"])
    loss = F.cross_entropy(outputs.logits, torch.tensor([label]))
    loss.backward()
    gradients = embeddings.grad.data

    influential_token_idx = gradients.abs().sum(dim=-1).argmax().item()
    influential_token = tokenizer.convert_ids_to_tokens([input_ids[0][influential_token_idx]])[0]

    tokens = tokenizer.tokenize(text)
    if influential_token in tokens:
        perturbed_tokens = [t if t != influential_token else "<perturbed>" for t in tokens]
        perturbed_text = tokenizer.convert_tokens_to_string(perturbed_tokens)
    else:
        perturbed_text = text
    return perturbed_text

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/HThesis/ham_attacked_non_attacked.csv", usecols=['text', 'label'])

# Create new data with original and attacked messages
new_data = []
for index, row in df.iterrows():
    original_message = row["text"]
    original_label = row["label"]
    new_data.append({"text": original_message, "label": original_label, "attack_type": "original"})
    attacked_message = gradient_based_attack(original_message, original_label)  # Assuming original_label is used for the attack
    new_data.append({"text": attacked_message, "label": 1, "attack_type": "attacked"})

# Create new DataFrame
attacked_df = pd.DataFrame(new_data)

# Save the dataset to a new CSV file for later use
df.to_csv("White_box_attacked_dataset.csv", index=False)

# Display the modified dataset
print(df)  # Or use your custom display function

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Split the dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# Tokenize and prepare data for training
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

train_dataset = TensorDataset(torch.tensor(train_encodings['input_ids']),
                             torch.tensor(train_encodings['attention_mask']),
                             torch.tensor(train_labels))
val_dataset = TensorDataset(torch.tensor(val_encodings['input_ids']),
                           torch.tensor(val_encodings['attention_mask']),
                           torch.tensor(val_labels))

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Set up optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Fine-tuning loop (example)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.train()  # Set the model to training mode

for epoch in range(3):  # Adjust the number of epochs as needed
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1} complete")

model.eval()  # Set the model back to evaluation mode



KeyError: 'label'